# 15 — Executar e verificar a avaliação da referência

Autor: Wanderson Ferreira. Preparação e atualização: 2026-10-09.

**HGB = Histogram-based Gradient Boosting**, modelo de abril já congelado.
O executor preserva o desenho da etapa 14 e não seleciona outro modelo.
A preparação usou fixtures sintéticas; posteriormente o autor executou e
verificou a avaliação de 02–15/09, sem refit ou novas runs.

**Estado atual:** avaliação, diagnóstico e reamostragem nativa revisados;
escopo exploratório concluído, com limites de cobertura explícitos.
As células abaixo inspecionam metadados e um bundle salvo, sem iniciar `run`.
Os resultados citados em Markdown são relatos do autor, não outputs inventados.


## Problema, alternativas e decisão

A janela foi fixada antes dos resultados. Precisamos verificar o modelo correto,
respeitar o histórico de rótulos, conservar todos os dias e registrar a abertura
da reserva. Reexecutar o pipeline original abriria períodos fora dessa autorização;
transformar a CLI de plano em avaliação também mudaria seu compromisso anterior.

Mantemos o plano de metadados e adicionamos um executor explícito com outro
snapshot de autorização. Reutilizamos o carregador de MLflow/skops, o SQL causal
e a interface de scores. O builder de dados fica em `data/datasets/`, as métricas
em `evaluation/` e a composição/auditoria em `modeling/experiments/`.

A assinatura opcional de raiz no carregador congelado permite apontar para o store
correto sem alterar o comportamento dos executores existentes. Não duplicamos
um trainer ou criamos outra implementação de features.


## Acesso e recuperação

| Fase | Compromisso |
| --- | --- |
| Conferir metadados | Inputs/código commitados, ambiente e fonte fixados |
| Carregar referência | Conferir bytes/signatura/parâmetros antes das partições |
| Construir features | Registrar acesso; ler somente 19/08–15/09 |
| Pontuar | Usar as 19 features dos 14 dias, sem fit ou novas runs |
| Publicar | Bundle atômico, relatório, predições, features e hashes |
| Verificar/retomar | Recalcular resultados salvos; não reabrir fonte/modelo quando completo |

`run` consome a reserva de qualidade de 02–15/09. Os dados não serão usados como
confirmação independente de hipóteses orientadas pelos novos resultados. O replay
16–30/09 continua fechado, e a fila permanece retrospectiva por dia completo.


In [ ]:
# Só metadados versionados; não executa a avaliação.
import json

from fraud_detection_mlops.config import PROJECT_ROOT

execution = json.loads(
    (PROJECT_ROOT / "references/reference_assessment_execution_v1.json").read_text()
)
execution["version"], execution["access_policy"]

## Checks da implementação e resultados fornecidos

Os cenários sintéticos percorreram 28 partições de contexto/avaliação, preservando
os 14 dias e verificando o histórico por máscara temporal independente. Partições
inválidas em maio e no replay não foram abertas. Dias de uma classe conservaram
contagens e desfechos indefinidos explicitamente nulos.

Os checks cobriram código não commitado, dados inválidos, symlinks, corrupção
rehasheada, concorrência e retomada após interrupção da publicação. A carga usou
MLflow/skops num store temporário. A suíte e os cenários estão no
[recibo da preparação](../../references/evidence/reference_assessment_execution_preparation_2026-10-09.json).

O autor executou e verificou a avaliação em `d2a40aac949fc273fab28f4ba40c86460097d06a`:
134.467 transações, AP **0,621312**, precisão média diária @100 de **54,93%** e
recall médio diário de **72,89%**, sem refit ou novas runs. São 769 capturas de
cliente-dia com fraude e 286 ocorrências fora da fila, entre 1.055; não são pessoas únicas.

Depois foram revisados o diagnóstico e a reamostragem circular de dias completos
com blocos 2/3/4/7 e 2.000 réplicas por configuração. Todas as métricas ficaram
definidas. Método, faixas e proveniência estão no
[resultado completo](../../docs/modeling/EVALUATION_PROTOCOL.md#resultado-nativo-da-reamostragem-e-fechamento)
e no [recibo nativo](../../references/evidence/reference_uncertainty_author_validation_2026-10-09.json).


In [ ]:
# Consulta apenas um bundle já publicado pelo executor; sem Silver ou MLflow.
from fraud_detection_mlops.modeling.experiments.reference_assessment_execution import (
    verify_assessment,
)

assessment_path = (
    PROJECT_ROOT
    / "data/processed/handbook"
    / "6e67dbd0a3bfe0d7ec33abc4bce5f37cd4ff0d6a"
    / execution["version"]
)
if (assessment_path / "results/manifest.json").is_file():
    saved = verify_assessment(assessment_path)
    print(
        {
            key: saved["metrics"][key]
            for key in ("rows", "average_precision", "daily_customer_precision_at_100")
        }
    )
else:
    print("Avaliação nativa ainda não publicada neste checkout.")

## Limites e próxima ação

A referência de fila aleatória condiciona nos rótulos/populações observados.
Seu intervalo não é incerteza de desempenho futuro, e sua cauda não autoriza
superioridade ou promoção. A maturação dos rótulos segue o relógio histórico:
os rótulos de 02–15/09 estão disponíveis antes de 23/09, mantendo causalidade
das features em cada transação.

Verificação offline confirma integridade/coerência dos resultados salvos; não
autentica externamente toda a execução. Os hashes fornecidos coincidiram com as
evidências anteriores e a implementação. Os bytes nativos das previsões não foram
recebidos pelo assistente; a AP reamostrada não foi recalculada independentemente.

O escopo exploratório está concluído. As faixas têm massa nominal de 95%, com
`coverage_validated: false`; não houve teste temporal formal, confirmação de
candidato ou promoção. A janela 02–15/09 foi consumida. 16–30/09 permanece
reservada para replay e não deve ser aberta por uma atualização documental.

Esta análise cumpriu o escopo exploratório: descreveu a utilidade da triagem na população observada, examinou a dependência dos dados e quantificou a sensibilidade das métricas.

| Achado | Benefício concreto |
| --- | --- |
| A fila capturou **769 de 1.055 ocorrências de cliente/dia com fraude**, deixando 286 fora | Descreve as capturas e perdas sob o orçamento de 100 alertas diários |
| A fila uniforme teria **28,3 capturas esperadas** nas populações e orçamentos observados | Quantifica a diferença para a política aleatória declarada nessa janela |
| Os limites das faixas exploratórias dos quatro comprimentos de bloco ficaram aproximadamente em **0,579–0,667 para AP** e **53,1%–56,6% para precisão @100** | Descreve sensibilidade à composição dos dias; não constitui um único intervalo de generalização |
| Clientes se repetem e há autocorrelação nas métricas | Sustenta o cuidado de não tratar 134.467 transações como observações independentes |

Agora temos uma referência com desempenho, sensibilidade, procedência e limitações documentados para avançar no laboratório de MLOps.

O autor já exportou a referência com 67.255 linhas de paridade e validou HTTP/wheel fora do checkout, preservando identidade e score de controle. O [relato nativo](../../references/evidence/laboratory_serving_native_validation_2026-10-09.json) registra essas saídas.

Próxima ação: revisar e integrar o PR desta etapa após validação e CI aprovadas. Em seguida, construir e medir Docker em uma branch a partir de main atualizada; depois Prefect e replay causal, conforme o [mural](../../docs/project/ROADMAP.md#próxima-entrega-concreta).
[Comandos e recuperação da avaliação](../../docs/modeling/EVALUATION_PROTOCOL.md#executor-auditável-da-janela-fixada).
